In [1]:
from reconstruction import *
from tqdm import tqdm
%matplotlib qt

In [2]:
#path_sm = 'Z:\\DATA\\4polar_data_raw\\2026_02_17_actin_Moein\\droplet3\\fov1\\sm\\Calib_Polar_2026-02-17\\images\\RAW_DATA\\image_Pos0.ome.tif'
path_reconstruction = 'E:\\2026_04_30_calibration\\beads\\Calib_Polar_2026-04-30\\images\\RAW_DATA\\reconstruction' 
path_beads = 'E:\\2026_04_30_calibration\\beads\\Calib_Polar_2026-04-30\\images\\RAW_DATA\\image_Pos0.ome.tif'
path_intensity = 'E:\\2026_04_30_calibration\\intensity\\Calib_Polar_2026-04-30\\images\\RAW_DATA\\image_Pos0.ome.tif'

In [4]:
data_beads = extract_raw_ome(path_beads)
data_intensity = extract_raw_ome(path_intensity)[0]
data = np.max(data_beads, axis=0)

In [5]:
data.shape

(512, 512)

In [6]:
data_beads.shape

(41, 512, 512)

In [7]:
data_intensity.shape

(512, 512)

In [8]:
x_0, x_1, x_2, x_3, x_4, x_5 = detect_plateau_0(data_intensity)
y_0, y_1, y_2, y_3 = detect_plateau_1(data_intensity)
plt.imshow(data_intensity)
plt.axvline(x_0, c='r')
plt.axvline(x_1, c='r')
plt.axvline(x_2, c='r')
plt.axvline(x_3, c='r')
plt.axvline(x_4, c='r')
plt.axvline(x_5, c='r')
plt.axhline(y_0, c='r')
plt.axhline(y_1, c='r')
plt.axhline(y_2, c='r')
plt.axhline(y_3, c='r')

In [9]:
read_noise = np.mean(data_intensity[:,1:y_0-2])
norm0 = np.mean(data_intensity[x_0:x_1,y_0:y_1])-read_noise
norm1 = np.mean(data_intensity[x_0:x_1,y_2:y_3])-read_noise
norm2 = np.mean(data_intensity[x_2:x_3,y_0:y_1])-read_noise
norm3 = np.mean(data_intensity[x_2:x_3,y_2:y_3])-read_noise
norm4 = np.mean(data_intensity[x_4:x_5,y_0:y_1])-read_noise
norm5 = np.mean(data_intensity[x_4:x_5,y_2:y_3])-read_noise

In [10]:
mn = (norm0+norm1+norm2+norm3+norm4+norm5)/6
norm = np.array([norm0/mn, norm1/mn, norm2/mn, norm3/mn, norm4/mn, norm5/mn])
print(norm)

[0.89114575 0.88772367 1.23625892 1.23217865 0.87478218 0.87791084]


In [11]:
%matplotlib qt
points = select_points_subimage(data, x_0, x_1, y_0, y_1)

In [12]:
points.shape

(5, 2)

In [13]:
v1 = [x_2-x_0, 0]
v2 = [x_4-x_0, 0]
v3 = [0, y_2-y_0]
v4 = [x_2-x_0, y_2-y_0]
v5 = [x_4-x_0, y_2-y_0]

In [14]:
p_list = fit_one_image(data, points)
p_list1 = fit_one_image(data, points+v1)
p_list2 = fit_one_image(data, points+v2)
p_list3 = fit_one_image(data, points+v3)
p_list4 = fit_one_image(data, points+v4)
p_list5 = fit_one_image(data, points+v5)

C:\Users\Amaury\Documents\GitHub\4polarMFM_these\reconstruction.py:93: OptimizeWarning: Covariance of the parameters could not be estimated
  p, pcov = curve_fit(gauss, xdata, zdata, p0=(70, points[i,0], points[i,1], 2, 150))


In [15]:
sim=np.zeros((512,512))
X, Y = np.meshgrid(np.arange(512), np.arange(512))
for p in p_list:
    sim += gauss((X, Y), p[0], p[1], p[2], p[3], p[4])
for p in p_list1:
    sim += gauss((X, Y), p[0], p[1], p[2], p[3], p[4])
for p in p_list2:
    sim += gauss((X, Y), p[0], p[1], p[2], p[3], p[4])
for p in p_list3:
    sim += gauss((X, Y), p[0], p[1], p[2], p[3], p[4])
for p in p_list4:
    sim += gauss((X, Y), p[0], p[1], p[2], p[3], p[4])
for p in p_list5:
    sim += gauss((X, Y), p[0], p[1], p[2], p[3], p[4])

In [17]:
%matplotlib qt
fig, ax = plt.subplots(1,2)
ax[0].imshow(sim, cmap='gray')
ax[1].imshow(data, cmap='gray')
plt.show()

In [ ]:
mux, muy, sigma = np.zeros((data_beads.shape[0]-16, 6, len(points))), np.zeros((data_beads.shape[0]-16, 6, len(points))), np.zeros((data_beads.shape[0]-16, 6, len(points)))
for i in tqdm(range(data_beads.shape[0]-16)):
    data = data_beads[i+8]
    p_list = fit_one_image(data, points)
    p_list1 = fit_one_image(data, points+v1)
    p_list2 = fit_one_image(data, points+v2)
    p_list3 = fit_one_image(data, points+v3)
    p_list4 = fit_one_image(data, points+v4)
    p_list5 = fit_one_image(data, points+v5)
    mux[i,0] = p_list[:,1]
    muy[i,0] = p_list[:,2]
    sigma[i,0] = p_list[:,3]
    mux[i,1] = p_list1[:,1]
    muy[i,1] = p_list1[:,2]
    sigma[i,1] = p_list1[:,3]
    mux[i,2] = p_list2[:,1]
    muy[i,2] = p_list2[:,2]
    sigma[i,2] = p_list2[:,3]
    mux[i,3] = p_list3[:,1]
    muy[i,3] = p_list3[:,2]
    sigma[i,3] = p_list3[:,3]
    mux[i,4] = p_list4[:,1]
    muy[i,4] = p_list4[:,2]
    sigma[i,4] = p_list4[:,3]
    mux[i,5] = p_list5[:,1]
    muy[i,5] = p_list5[:,2]
    sigma[i,5] = p_list5[:,3]

In [ ]:
%matplotlib inline
plt.plot(np.mean(sigma, axis=2)[:,0])

In [ ]:
v1_ = np.array([np.mean(mux[15:25,1]-mux[15:25,0]), np.mean(muy[15:25,1]-muy[15:25,0])])
v2_ = np.array([np.mean(mux[15:25,2]-mux[15:25,0]), np.mean(muy[15:25,2]-muy[15:25,0])])
v3_ = np.array([np.mean(mux[15:25,3]-mux[15:25,0]), np.mean(muy[15:25,3]-muy[15:25,0])])
v4_ = np.array([np.mean(mux[15:25,4]-mux[15:25,0]), np.mean(muy[15:25,4]-muy[15:25,0])])
v5_ = np.array([np.mean(mux[15:25,5]-mux[15:25,0]), np.mean(muy[15:25,5]-muy[15:25,0])])

In [ ]:
v1 = np.round(v1_).astype(int)
v2 = np.round(v2_).astype(int)
v3 = np.round(v3_).astype(int)
v4 = np.round(v4_).astype(int)
v5 = np.round(v5_).astype(int)
dv1 = v1_-v1
dv2 = v2_-v2
dv3 = v3_-v3
dv4 = v4_-v4
dv5 = v5_-v5

In [ ]:
H = x_1-x_0
L = y_1-y_0

In [ ]:
indices = np.arange(7999)
os.makedirs(path_reconstruction, exist_ok=True)

In [ ]:
%matplotlib inline
for i in tqdm(indices):
    data_sm = extract_raw_ome(path_sm, i, i+1)[0]
    #plt.imshow(data_sm)
    #plt.show()
    stack = np.zeros((6,L,H))
    stack[0] = data_sm[y_0:y_0+L,x_0:x_0+H]
    stack[1] = data_sm[y_0+v3[1]:y_0+v3[1]+L,x_0+v3[0]:x_0+v3[0]+H]
    stack[2] = data_sm[y_0+v1[1]:y_0+v1[1]+L,x_0+v1[0]:x_0+v1[0]+H]
    stack[3] = data_sm[y_0+v4[1]:y_0+v4[1]+L,x_0+v4[0]:x_0+v4[0]+H]
    stack[4] = data_sm[y_0+v2[1]:y_0+v2[1]+L,x_0+v2[0]:x_0+v2[0]+H]
    stack[5] = data_sm[y_0+v5[1]:y_0+v5[1]+L,x_0+v5[0]:x_0+v5[0]+H]
    stack = stack*norm[:,None,None]
    tifffile.imwrite(path_reconstruction+'\\frame_'+np.char.zfill(str(i), 5), stack)